# 03 — HRV Feature Engineering
**NANO Study | ESD Lab, University of South Carolina**

> ⚠️ **HIPAA Notice**: Synthetic data only.

## Goals
1. Extract time-domain HRV features (SDNN, RMSSD, CVNN, HTI)
2. Compute Poincaré plot features (SD1, SD2)
3. Estimate RSA via continuous wavelet transform
4. Identify HDA phases from HR difference waveforms
5. Visualize all features with publication-quality plots

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import signal as scipy_signal
from scipy.stats import entropy

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

rng = np.random.default_rng(seed=42)
print('Setup complete.')

## 1. Generate Synthetic IBI Series

In [ ]:
# Simulate realistic preterm infant IBI series (~140 bpm mean HR)
n_beats = 500
mean_ibi = 428.0   # ms (140 bpm)
ibis = mean_ibi + rng.normal(0, 20, n_beats)  # SDNN ~ 20 ms
# Add respiratory modulation for RSA (~1.5 Hz for preterm)
t_beats = np.cumsum(ibis) / 1000.0
resp_freq = 0.8  # Hz (slightly below typical preterm range)
ibis += 15 * np.sin(2 * np.pi * resp_freq * t_beats[:n_beats])
# Clip to physiologically plausible range
ibis = np.clip(ibis, 300, 1500)
print(f'Generated {n_beats} IBIs | Mean: {ibis.mean():.1f} ms | SD: {ibis.std():.1f} ms')

## 2. Time-Domain HRV Features

In [ ]:
def compute_timedomain_hrv(ibis: np.ndarray) -> dict:
    """Compute standard time-domain HRV features."""
    valid = ibis[~np.isnan(ibis)]
    diff_ibis = np.diff(valid)
    n_bins = max(8, int(np.sqrt(len(valid))))
    hist, _ = np.histogram(valid, bins=n_bins)
    hti = len(valid) / np.max(hist) if np.max(hist) > 0 else np.nan
    return {
        'mean_ibi_ms':  round(float(np.mean(valid)), 3),
        'mean_hr_bpm':  round(float(60000 / np.mean(valid)), 3),
        'sdnn_ms':      round(float(np.std(valid, ddof=1)), 3),
        'rmssd_ms':     round(float(np.sqrt(np.mean(diff_ibis**2))), 3),
        'cvnn_pct':     round(float(100 * np.std(valid, ddof=1) / np.mean(valid)), 3),
        'hti':          round(float(hti), 3),
        'nn50':         int(np.sum(np.abs(diff_ibis) > 50)),
        'pnn50':        round(float(100 * np.sum(np.abs(diff_ibis) > 50) / len(diff_ibis)), 3),
    }

hrv_td = compute_timedomain_hrv(ibis)
pd.Series(hrv_td).rename('Value').to_frame()

## 3. Poincaré Plot Features (SD1, SD2)

In [ ]:
def poincare_features(ibis: np.ndarray) -> dict:
    """Compute SD1, SD2 and their ratio from the Poincaré plot."""
    valid = ibis[~np.isnan(ibis)]
    x = valid[:-1]
    y = valid[1:]
    sd1 = np.sqrt(0.5 * np.var(y - x, ddof=1))
    sd2 = np.sqrt(2 * np.var(x, ddof=1) - 0.5 * np.var(y - x, ddof=1))
    return {'sd1_ms': round(float(sd1), 3),
            'sd2_ms': round(float(sd2), 3),
            'sd1_sd2_ratio': round(float(sd1 / sd2) if sd2 > 0 else np.nan, 3)}

pp = poincare_features(ibis)
print(f"Poincaré SD1: {pp['sd1_ms']:.2f} ms | SD2: {pp['sd2_ms']:.2f} ms | Ratio: {pp['sd1_sd2_ratio']:.3f}")

# Poincaré plot
valid = ibis[~np.isnan(ibis)]
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(valid[:-1], valid[1:], alpha=0.3, s=5, color='steelblue')
ax.set_xlabel('IBI$_n$ (ms)')
ax.set_ylabel('IBI$_{n+1}$ (ms)')
ax.set_title(f'Poincaré Plot\nSD1={pp["sd1_ms"]:.1f} ms, SD2={pp["sd2_ms"]:.1f} ms')
ax.set_aspect('equal')
ax.plot([300, 700], [300, 700], 'k--', alpha=0.3, lw=0.8, label='Identity')
ax.legend()
plt.tight_layout()
plt.show()

## 4. RSA via Continuous Wavelet Transform

RSA (Respiratory Sinus Arrhythmia) is estimated as log-transformed power in the respiratory frequency band. For VPT infants we use 0.2–1.5 Hz (adjusted from adult 0.15–0.40 Hz).

In [ ]:
def estimate_rsa_cwt(ibis: np.ndarray, fs_resample: float = 4.0,
                      resp_low: float = 0.2, resp_high: float = 1.5) -> float:
    """Estimate RSA as log(power) in respiratory band via Welch PSD."""
    valid = ibis[~np.isnan(ibis)]
    t_orig = np.cumsum(valid) / 1000.0
    t_new = np.arange(0, t_orig[-1], 1.0 / fs_resample)
    ibi_resampled = np.interp(t_new, t_orig, valid)
    freqs, psd = scipy_signal.welch(ibi_resampled, fs=fs_resample, nperseg=64)
    resp_mask = (freqs >= resp_low) & (freqs <= resp_high)
    resp_power = np.trapz(psd[resp_mask], freqs[resp_mask])
    rsa = float(np.log(resp_power)) if resp_power > 0 else np.nan
    return rsa

rsa_val = estimate_rsa_cwt(ibis)
print(f'RSA (log power, 0.2–1.5 Hz): {rsa_val:.4f}')

# Show PSD
valid = ibis[~np.isnan(ibis)]
t_orig = np.cumsum(valid) / 1000.0
t_new = np.arange(0, t_orig[-1], 0.25)
ibi_rs = np.interp(t_new, t_orig, valid)
freqs, psd = scipy_signal.welch(ibi_rs, fs=4.0, nperseg=64)

fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(freqs, psd, color='royalblue', lw=1.5)
ax.axvspan(0.2, 1.5, alpha=0.15, color='green', label='RSA band (0.2–1.5 Hz)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (ms²/Hz)')
ax.set_title('HRV Power Spectral Density (Welch)')
ax.legend()
ax.set_xlim(0, 2.0)
plt.tight_layout()
plt.show()

## 5. HDA Phase Identification

Heart rate deceleration (HDA) phases are identified from HR difference waveforms time-locked to behavioral event onsets (looking events from DataVyu).

In [ ]:
# Simulate an HDA-like HR response: baseline → decel (sustained attention) → return
t_hda = np.linspace(-5, 15, 500)
hr_baseline = 140.0
# HDA phases: orienting (0-2s, -8 bpm), sustained attention (2-8s, -15 bpm),
# termination (8-11s, return), inattention (11-15s, +3 bpm)
hr_response = hr_baseline * np.ones(len(t_hda))
orient = (t_hda >= 0) & (t_hda < 2)
sa = (t_hda >= 2) & (t_hda < 8)
term = (t_hda >= 8) & (t_hda < 11)
inatt = (t_hda >= 11)
hr_response[orient] -= 8 * (t_hda[orient] / 2)
hr_response[sa] = hr_baseline - 15
hr_response[term] = hr_baseline - 15 + 15 * ((t_hda[term] - 8) / 3)
hr_response[inatt] = hr_baseline + 3
hr_response += rng.normal(0, 1.5, len(t_hda))

colors = {'Baseline': 'gray', 'Orienting': 'steelblue',
          'Sustained\nAttention': 'forestgreen', 'Termination': 'darkorange',
          'Inattention': 'firebrick'}
phase_bounds = [(-5, 0), (0, 2), (2, 8), (8, 11), (11, 15)]
phase_names  = ['Baseline', 'Orienting', 'Sustained\nAttention', 'Termination', 'Inattention']

fig, ax = plt.subplots(figsize=(10, 4))
for (lo, hi), name, c in zip(phase_bounds, phase_names, colors.values()):
    mask = (t_hda >= lo) & (t_hda < hi)
    ax.axvspan(lo, hi, alpha=0.12, color=c)
    ax.text((lo + hi) / 2, 127, name, ha='center', fontsize=8, color=c)
ax.plot(t_hda, hr_response, lw=1.5, color='navy')
ax.axvline(0, color='black', lw=1, linestyle='--', label='Event onset')
ax.axhline(hr_baseline, color='gray', lw=0.8, linestyle=':')
ax.set_xlabel('Time relative to look onset (s)')
ax.set_ylabel('Heart Rate (bpm)')
ax.set_title('Simulated HDA Phase Response — Preterm Infant')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

| Feature | Value |
|---------|-------|
| Mean IBI (ms) | See output above |
| SDNN (ms) | See output above |
| RMSSD (ms) | See output above |
| SD1 / SD2 | See output above |
| RSA (log power) | See output above |

**Next**: See `04_imputation_analysis.ipynb` for MICE workflow.